# Cleaning our Datasets

In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from helpers import column_utils, csv_utils, merge_utils

In [2]:
df_draft = csv_utils.csv_read('../data/nfl_draft_data.csv', ';')


df_ras = csv_utils.csv_read('../data/RAS_1987_present.csv', ',')

## To start, we need to perform some cleaning tasks on the columns we will merge on. These include:
- Player Name
- Team Name
- Draft Round


***

### Player Name

In [3]:
df_draft['name'] = column_utils.column_remove_pattern(
    column_utils.column_strip_whitespace(
        column_utils.column_to_lowercase(
            df_draft['name']
        )
    ),
    column_utils.REGEX_NAME_PATTERN,
    ''
)


df_ras['Name'] = column_utils.column_remove_pattern(
    column_utils.column_strip_whitespace(
        column_utils.column_to_lowercase(
            df_ras['Name']
        )
    ),
    column_utils.REGEX_NAME_PATTERN,
    ''
)

### Team Name



We want to only include the mascot name in the team name. Our RAS dataset already has this, so we will just strip whitespace and move it to lowercase. Our Draft dataset will require us to do a few other things

- Replace historical team names with their current variant
    - Some team names will be straightforward to deal with even if they are the old team name. These include names like the St. Louis Rams, and the Los Angeles Rams. This is the same franchise, so using just the mascot here is a simple change. 
    - The Washington Commanders however have had other completely different mascots in their past, we will fix this by replacing instances of their old team names (*ex. washington football team*)
    - We can then apply our function to remove the city from the team name and only keep the mascot. We will also apply the same whitespace trim and to lowercase function.

In [4]:
df_draft['team'] = column_utils.column_replace_value(
    df_draft['team'],
    column_utils.WASHINGTON_TEAM_NAMES,
    'commanders'
)


df_draft['team'] = column_utils.column_replace_value(
    df_draft['team'],
    column_utils.HOUSTON_TEAM_NAMES,
    'titans'
)

In [5]:
df_draft['team'] = column_utils.column_strip_whitespace(
    column_utils.column_to_lowercase(
        column_utils.column_remove_pattern(
            df_draft['team'],
            column_utils.REGEX_TEAM_PATTERN,
            r'\1'
        )
    )
)


df_ras['Draft Team'] = column_utils.column_strip_whitespace(
    column_utils.column_to_lowercase(
        df_ras['Draft Team']
    )
)

### Draft Round

In [6]:
df_draft['draft_round'] = column_utils.column_convert_value(
    df_draft['draft_round'], 'Int64'
)

***

# Filling in Some Merge Key Values

We know the columns we will merge our two datasets on will be

- Player Name
- Team Name
- Draft Round
- Draft Year

We are going to want to ensure these values are filled in for both datasets to ensure the merge goes smoothly. 

In [7]:
print('Missing Merge Key values in Draft Dataset\n')
print(df_draft[['name', 'team', 'draft_round', 'year']].isna().sum())
print('- - - - - - - - - - - - - - - - - ')
print('Missing Merge Key values in RAS Dataset\n')
print(df_ras[['Name', 'Draft Team', 'Round', 'Year']].isna().sum())

Missing Merge Key values in Draft Dataset

name              0
team           5929
draft_round    5929
year              0
dtype: int64
- - - - - - - - - - - - - - - - - 
Missing Merge Key values in RAS Dataset

Name          0
Draft Team    0
Round         0
Year          0
dtype: int64


Let's see if we can fill in these `team` and `draft_round` values for our draft dataset using the player's information from the RAS dataset

We know we wont be able to fill in all the missing values due to the discrepancy in dataset size and context.

The draft dataset has over 14,000 rows while the RAS dataset has about 9,700. so there will still be players that are in one dataset but not the other

In [8]:
debug = merge_utils.merge_datasets(
    df_draft,
    df_ras,
    ['name', 'team', 'draft_round', 'year'],
    ['Name', 'Draft Team', 'Round', 'Year'],
    'outer', 
    None,
    True
)

In [9]:
names_from_draft = debug[(debug['_merge'] == 'left_only')]['name'].to_numpy()


names_from_draft_in_ras_unmerged = df_ras[
    df_ras['Name'].isin(names_from_draft)][
        ['Name', 'Draft Team', 'Round', 'Year']
]


count = len(names_from_draft_in_ras_unmerged)


msg = (
    'Number of players in the draft dataset who appear in the RAS '
    f'Dataset but could not be merged: {count}'
)


print(msg)

Number of players in the draft dataset who appear in the RAS Dataset but could not be merged: 685


We can build a dictionary to fill these values. We will need a key column on both dataframes and then we can convert a series into a dictionary. This dictionary can then be mapped to the rows we have missing values for

In [10]:
# Removing whitespace from the merge key columns we are trying to replace
df_draft['name'] = column_utils.column_strip_whitespace(
    df_draft['name']
)
df_draft['team'] = column_utils.column_strip_whitespace(
    df_draft['team']
)
df_ras['Name'] = column_utils.column_strip_whitespace(
    df_ras['Name']
)
df_ras['Draft Team'] = column_utils.column_strip_whitespace(
    df_ras['Draft Team']
)


# using player and year drafted as the key to ensure uniqueness
df_draft['key'] = df_draft['name'] + "_" + df_draft['year'].astype(str)
df_ras['key'] = df_ras['Name'] + "_" + df_ras['Year'].astype(str)
df_draft['key'] = column_utils.column_strip_whitespace(
    df_draft['key']
)
df_ras['key'] = column_utils.column_strip_whitespace(
    df_ras['key']
)


# creating lookup dictionaries to fill in missing values in the draft dataset
team_lookup = df_ras.set_index('key')['Draft Team'].to_dict()
round_lookup = df_ras.set_index('key')['Round'].to_dict()


# mapping the missing values in the draft dataset using the lookup dictionaries
df_draft['team'] = df_draft['team'].fillna(
    df_draft['key'].map(team_lookup)
)


df_draft['draft_round'] = df_draft['draft_round'].fillna(
    df_draft['key'].map(round_lookup)
)

In [11]:
debug = merge_utils.merge_datasets(
    df_draft,
    df_ras,
    ['name', 'team', 'draft_round', 'year'],
    ['Name', 'Draft Team', 'Round', 'Year'],
    'outer',
    None,
    True
)


names_from_draft = debug[(debug['_merge'] == 'left_only')]['name'].to_numpy()


names_from_draft_in_ras_unmerged = df_ras[
    df_ras['Name'].isin(names_from_draft)][
        ['Name', 'Draft Team', 'Round', 'Year']
]


count = len(names_from_draft_in_ras_unmerged)
msg = (
    'Number of players in the draft dataset who appear in the RAS '
    f'Dataset but could not be merged: {count}'
)
print(msg)

Number of players in the draft dataset who appear in the RAS Dataset but could not be merged: 376


We were able to fill in some values for draft team and round

Filling in these missing merge key values will help in our merge, and we should see more data being populated in the merged dataset

***

# Merging Our Datasets

In [12]:
df_merged_draft_base_left = merge_utils.merge_datasets(
    df_draft,
    df_ras,
    ['name', 'team', 'draft_round', 'year'],
    ['Name', 'Draft Team', 'Round', 'Year'],
    'left',
    'm:1'
)


df_merged_ras_base_right = merge_utils.merge_datasets(
    df_draft,
    df_ras,
    ['name', 'team', 'draft_round', 'year'],
    ['Name', 'Draft Team', 'Round', 'Year'],
    'right',
    'm:1'
)

***

### Converting to CSV

In [13]:
csv_utils.csv_write(
    df_merged_draft_base_left, 
    '../data/cleaned/merged_draft_base_left_join.csv'
)


csv_utils.csv_write(
    df_merged_ras_base_right, 
    '../data/cleaned/merged_ras_base_right_join.csv'
)